In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from ipywidgets import Dropdown, interact
import os

# Read the CSV file
csv_path = 'ram_independence_results.csv'
df = pd.read_csv(csv_path)

# Group by commit and build_config, taking the mean of mlp_max_time_us
# This aggregates across different conditions (flooding, dormant, etc.)
agg_df = df.groupby(['git_commit', 'build_config'])['mlp_max_time_us'].mean().reset_index()
agg_df.columns = ['git_commit', 'build_config', 'mlp_max_time_us']

# Calculate the total (sum across all build_configs) for each commit
commit_totals = agg_df.groupby('git_commit')['mlp_max_time_us'].sum().reset_index()
commit_totals.columns = ['git_commit', 'total_mlp_max_time_us']

# Get unique commits in order of appearance (most recent first would require git)
# For now, we'll get them in order of appearance in the CSV
unique_commits = df['git_commit'].unique()

# Find the commit with the lowest total mlp_max_time_us
lowest_sum_commit = commit_totals.loc[commit_totals['total_mlp_max_time_us'].idxmin(), 'git_commit']

# Find the most recent commit (last in the CSV)
most_recent_commit = unique_commits[-1]

print(f"Commits available: {len(unique_commits)}")
print(f"Lowest sum commit: {lowest_sum_commit}")
print(f"Most recent commit: {most_recent_commit}")

Commits available: 8
Lowest sum commit: 17bc32a0f8233579d5a05abd14faf43953dc464b
Most recent commit: de666b60ed04c65b78353f502233da6f993f780d


In [2]:
from ipywidgets import Dropdown, HBox, VBox, Output
from IPython.display import clear_output

# Create dropdowns with default values
commit_options = {commit: commit for commit in unique_commits}

dropdown1 = Dropdown(options=commit_options, value=lowest_sum_commit, description='Commit 1:')
dropdown2 = Dropdown(options=commit_options, value=most_recent_commit, description='Commit 2:')

output = Output()

def plot_comparison(commit1, commit2):
    with output:
        clear_output(wait=True)
        
        # Get data for both commits from the original dataframe (with conditions)
        data1 = df[df['git_commit'] == commit1]
        data2 = df[df['git_commit'] == commit2]
        
        # Create the plot
        fig, ax = plt.subplots(figsize=(12, 6))
        
        build_configs = ['core0', 'core1', 'naive']
        x_positions = [0, 1, 2]
        
        colors_commit = ['#1f77b4', '#ff7f0e']  # Blue for commit1, Orange for commit2
        
        for idx, (commit, data, color) in enumerate([(commit1, data1, colors_commit[0]), (commit2, data2, colors_commit[1])]):
            # Get flooding and dormant values for each build config
            flooding_values = []
            dormant_values = []
            
            for bc in build_configs:
                flooding = data[(data['build_config'] == bc) & (data['condition'] == 'flooding')]['mlp_max_time_us'].values
                dormant = data[(data['build_config'] == bc) & (data['condition'] == 'dormant')]['mlp_max_time_us'].values
                
                flooding_values.append(flooding[0] if len(flooding) > 0 else 0)
                dormant_values.append(dormant[0] if len(dormant) > 0 else 0)
            
            # Offset x positions slightly for the two commits
            x_offset = x_positions if idx == 0 else [x + 0.25 for x in x_positions]
            
            # Plot area between dormant (bottom) and flooding (top) with transparency
            ax.fill_between(x_offset, dormant_values, flooding_values, alpha=0.4, color=color, label=f'{commit[:7]}')
        
        ax.set_xlabel('Build Config', fontsize=12)
        ax.set_ylabel('mlp_max_time_us', fontsize=12)
        ax.set_title('MLP Maximum Time Comparison (Dormant to Flooding)', fontsize=14)
        ax.set_xticks([0.125, 1.125, 2.125])
        ax.set_xticklabels(build_configs)
        ax.legend(loc='upper left', fontsize=10)
        ax.grid(True, alpha=0.3, axis='y')
        
        plt.tight_layout()
        plt.show()
        
        # Print numerical comparison
        print(f"\n{'Build Config':<12} {'Condition':<10} {'Commit 1':<20} {'Commit 2':<20}")
        print("-" * 62)
        for bc in build_configs:
            for cond in ['dormant', 'flooding']:
                val1 = df[(df['git_commit'] == commit1) & (df['build_config'] == bc) & (df['condition'] == cond)]['mlp_max_time_us'].values
                val2 = df[(df['git_commit'] == commit2) & (df['build_config'] == bc) & (df['condition'] == cond)]['mlp_max_time_us'].values
                v1_str = f"{val1[0]:.0f}" if len(val1) > 0 else "N/A"
                v2_str = f"{val2[0]:.0f}" if len(val2) > 0 else "N/A"
                print(f"{bc:<12} {cond:<10} {v1_str:<20} {v2_str:<20}")

def on_value_change(change):
    plot_comparison(dropdown1.value, dropdown2.value)

dropdown1.observe(on_value_change, names='value')
dropdown2.observe(on_value_change, names='value')

# Initial plot
plot_comparison(dropdown1.value, dropdown2.value)

# Display the interface
VBox([HBox([dropdown1, dropdown2]), output])